# 3 Sep

# Parsing and checking out the coconut_with_cids.csv so I can ensure completeness for the DB and remove things I dont need like the descriptors and the organisms

## Getting an idea of whats here what is missing and what is distinct

In [1]:
import pandas as pd

coco = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', low_memory=False)

columns = coco.columns
columns.to_list()

entries = []
for col in columns:
    num_entries = len(coco[col])
    missing = coco[col].isnull().sum()
    distinct = coco[col].nunique()
    entries.append([col,num_entries,missing,distinct])

df_out = pd.DataFrame(entries,columns=['column_name','number_of_entries','missing_entries','distinct_entries'])
print(df_out)

                         column_name  number_of_entries  missing_entries  \
0                         identifier             738827                0   
1                   canonical_smiles             738827                0   
2                     standard_inchi             738827                0   
3                 standard_inchi_key             738827                0   
4                               name             738827           375404   
5                         iupac_name             738827            74593   
6                   annotation_level             738827                0   
7                   total_atom_count             738827                0   
8                   heavy_atom_count             738827                0   
9                   molecular_weight             738827                0   
10            exact_molecular_weight             738827                0   
11                 molecular_formula             738827                0   
12          

## Checking the coconut paper (old paper to be fair) claim that all compounds have a name or iupac

In [1]:
import pandas as pd

coco = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', low_memory=False)
count = 0
for idx, row in coco.iterrows():
    if pd.isna(row.name) and pd.isna(row.iupac_name) and pd.isna(row.synonyms):
        count +=   1

print(f'There are {count} entries with no obvious name')

There are 0 entries with no obvious name


## Giving each compound a name and dropping some of the columns I don't need

In [3]:
# im lazy just printing column names so I can copy paste
import pandas as pd

coco = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', low_memory=False)

cols = coco.columns
cols = cols.to_list()
print(cols)

['identifier', 'canonical_smiles', 'standard_inchi', 'standard_inchi_key', 'name', 'iupac_name', 'annotation_level', 'total_atom_count', 'heavy_atom_count', 'molecular_weight', 'exact_molecular_weight', 'molecular_formula', 'alogp', 'topological_polar_surface_area', 'rotatable_bond_count', 'hydrogen_bond_acceptors', 'hydrogen_bond_donors', 'hydrogen_bond_acceptors_lipinski', 'hydrogen_bond_donors_lipinski', 'lipinski_rule_of_five_violations', 'aromatic_rings_count', 'qed_drug_likeliness', 'formal_charge', 'fractioncsp3', 'number_of_minimal_rings', 'van_der_walls_volume', 'contains_sugar', 'contains_ring_sugars', 'contains_linear_sugars', 'murcko_framework', 'np_likeness', 'chemical_class', 'chemical_sub_class', 'chemical_super_class', 'direct_parent_classification', 'np_classifier_pathway', 'np_classifier_superclass', 'np_classifier_class', 'np_classifier_is_glycoside', 'organisms', 'collections', 'dois', 'synonyms', 'cas', 'coconut_id', 'cid']


In [4]:
import pandas as pd

coco = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', usecols=['identifier', 'canonical_smiles', 'standard_inchi', 'standard_inchi_key', 'name', 'iupac_name',
                                                                                               'np_likeness','collections', 'dois', 'synonyms', 'cas', 'coconut_id', 'cid'], low_memory=False)
count = 0
for idx, row in coco.iterrows():
    if pd.isna(row.name) and pd.isna(row.iupac_name):
        count += 1
print(count)


0


## for the above I was going to sort out naming but apparently no entry is missing its iupac and name but it looks like the iupacs missinga are not necessarily all unresolveable so that is the job here but I am writing this so it can be done on wonko

- about 26% of the compounds that do not have an IUPAC do have a CID so I will just send the ones with cids to wonko to try
- first code block generates the file for wonko the second will be the actual script 

In [1]:
import pandas as pd

coco = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', usecols=['identifier', 'iupac_name', 'cid'], low_memory=False)

df = coco[coco["iupac_name"].isna() | coco["iupac_name"].astype(str).str.strip().eq("")].copy()

df.to_csv('/home/school/masters/Scripts/coconut_full/coconut_compounds_need_iupac_for_wonko.csv',index=False)

In [ ]:
import pandas as pd
import time
import pubchempy as pcp

def get_iupac(cid):
    for k in range(5):
        try:
            compound = pcp.get_compounds(int(cid),'cid')
            if compound:
                iupac = compound[0].iupac_name
                if iupac:
                    return iupac, None
                else:
                    return None, 'no iupac'
            else: 
                return None, 'didnt work not on pubchem'
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or '503' in error_msg or '504' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower():
                time.sleep(5)
            else:
                return None, str(e)
    return None, 'retry uncsucessful'

def progress_bar(count, total):
    bar_length = 40
    filled_length = int(bar_length * count // total)
    bar = '█' * filled_length + '-' * (bar_length - filled_length)
    out = f'\rProgress: |{bar}| {count}/{total} ({(count/total)*100:.2f}%)'
    return out

in_path = '/nlustre/users/nathanc/coconut_compounds_need_iupac_for_wonko.csv'
results_path  = '/nlustre/users/nathanc/coconut_now_with_iupac_from_pubchem.csv'
error_path = '/nlustre/users/nathanc/iupac_getter_errors.csv'

df = pd.read_csv(in_path)
err = []
count = 0
length = len(df)

for idx, row in df.iterrows():
        if pd.notna(row.cid):
            time.sleep(0.2)
            cid = int(row.cid) 
            
            iupac, errorz = get_iupac(cid)

            if iupac:
                df.at[idx,'iupac_name'] = iupac
            else:
                err.append([cid,errorz])
        count += 1
        
        if count % 1000 == 0:
                df.to_csv(results_path,index=False)
                errors = pd.DataFrame(err, columns = ['cid','err'])
                errors.to_csv(error_path, index=False)
        print(progress_bar(count,length))

df.to_csv(results_path, index=False)
errors = pd.DataFrame(err, columns = ['cid','err'])
errors.to_csv(error_path, index=False)


# 4 Sep

## Wonko made a nice file and error file for the iupac getter stuff. (coconut_full/iupac_getter_errors.csv,coconut_full/coconut_now_with_iupac_from_pubchem.csv)
- just checking the error file to see whats up

In [ ]:
The below is accidental copy paste see labbook for output
import pandas as pd
import time
import pubchempy as pcp

def get_iupac(cid):
    for k in range(5):
        try:
            compound = pcp.get_compounds(int(cid),'cid')
            if compound:
                iupac = compound[0].iupac_name
                if iupac:
                    return iupac, None
                else:
                    return None, 'no iupac'
            else: 
                return None, 'didnt work not on pubchem'
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or '503' in error_msg or '504' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower():
                time.sleep(5)
            else:
                return None, str(e)
    return None, 'retry uncsucessful'

def progress_bar(count, total):
    bar_length = 40
    filled_length = int(bar_length * count // total)
    bar = '█' * filled_length + '-' * (bar_length - filled_length)
    out = f'\rProgress: |{bar}| {count}/{total} ({(count/total)*100:.2f}%)'
    return out

in_path = '/nlustre/users/nathanc/coconut_compounds_need_iupac_for_wonko.csv'
results_path  = '/nlustre/users/nathanc/coconut_now_with_iupac_from_pubchem.csv'
error_path = '/nlustre/users/nathanc/iupac_getter_errors.csv'

df = pd.read_csv(in_path)
err = []
count = 0
length = len(df)

for idx, row in df.iterrows():
        if pd.notna(row.cid):
            time.sleep(0.2)
            cid = int(row.cid) 
            
            iupac, errorz = get_iupac(cid)

            if iupac:
                df.at[idx,'iupac_name'] = iupac
            else:
                err.append([cid,errorz])
        count += 1
        
        if count % 1000 == 0:
                df.to_csv(results_path,index=False)
                errors = pd.DataFrame(err, columns = ['cid','err'])
                errors.to_csv(error_path, index=False)
        print(progress_bar(count,length))

df.to_csv(results_path, index=False)
errors = pd.DataFrame(err, columns = ['cid','err'])
errors.to_csv(error_path, index=False)




2537
                                            error_type  count
0                                             no iupac   2519
169                                 retry uncsucessful     15
215  <urlopen error [Errno -2] Name or service not ...      3


# 4/7 sep

## retrying the 18 that didn't come back as no iupac

In [8]:
import pandas as pd
import time
import pubchempy as pcp

def get_iupac(cid):
    for k in range(5):
        try:
            compound = pcp.get_compounds(int(cid),'cid')
            if compound:
                iupac = compound[0].iupac_name
                if iupac:
                    return iupac, None
                else:
                    return None, 'no iupac'
            else: 
                return None, 'didnt work not on pubchem'
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or '503' in error_msg or '504' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower():
                time.sleep(5)
            else:
                return None, str(e)
    return None, 'retry uncsucessful'

in_path = '/home/school/masters/Scripts/coconut_full/iupac_getter_errors.csv'

df = pd.read_csv(in_path)
new_iupac = []
count = 0
length = len(df)

for idx, row in df.iterrows():
        if row.err != 'no iupac':
            time.sleep(0.2)
            cid = int(row.cid) 
            
            iupac, errorz = get_iupac(cid)
            if iupac:
                new_iupac.append([cid, iupac])                
                print([cid,iupac])
            else:
                print([cid,errorz])
        count += 1

new_iupac = pd.DataFrame(new_iupac,columns=['cid','iupac_name'])
df_old_iupac = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_now_with_iupac_from_pubchem.csv')

iupac_map = new_iupac.set_index('cid')['iupac_name']

df_old_iupac['iupac_name'] = df_old_iupac['iupac_name'].fillna(df_old_iupac['cid'].map(iupac_map))

df_old_iupac.to_csv('/home/school/masters/Scripts/coconut_full/coconut_now_with_iupac_from_pubchem.csv',index= False)


[87443963, '(6E,10E,14E,16E,18E,20E,22E,26E)-31-methoxy-2,6,10,14,19,23,27,31-octamethyldotriaconta-6,10,14,16,18,20,22,26-octaen-2-ol']
[10754966, '(NE)-N-(12-pyridin-3-yldodecylidene)hydroxylamine']
[92033736, '(Z)-N-(2-methylpropyl)non-2-en-6,8-diynamide']
[164449732, '[2-[(Z)-hexadec-7-enoyl]oxy-3-[hydroxy-[2,3,4,5-tetrahydroxy-6-[3,4,5-trihydroxy-6-(hydroxymethyl)oxan-2-yl]oxycyclohexyl]oxyphosphoryl]oxypropyl] (Z)-octadec-11-enoate']
[135601000, '(NZ)-N-[(16E)-16-hydroxyimino-13-methyl-3-prop-2-enoxy-6,9,11,12,14,15-hexahydrocyclopenta[a]phenanthren-17-ylidene]hydroxylamine']
[90478576, 'N-[(E)-3-(2-amino-1H-imidazol-5-yl)prop-2-enyl]-4,5-dibromo-1H-pyrrole-2-carboxamide;hydrochloride']
[5388464, '(4Z)-4-[(2-fluorophenyl)methylidene]-2-methyl-1,3-oxazol-5-one']
[89042540, '[(Z)-3-(4-methoxyphenyl)-4-phenylbut-2-en-2-yl] 4-methylbenzenesulfonate']
[44575966, '[(2S,3R,4S,5S)-5-hydroxy-2-[[(3S,8R,9S,10R,13S,14S,16S,17S)-17-hydroxy-10,13-dimethyl-17-[(2S)-6-methyl-3-oxoheptan-2-yl]-3

# 7 Sep

## now that I have all the iupacs that are listed by pubchem but were blank in the coconut
- going to do some stats to see what was added and what existed
- going to add the new iupacs

In [ ]:
import pandas as pd

df_old_less_iupac = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv')

total_compounds = len(df_old_less_iupac)
number_of_iupac_missing = df_old_less_iupac['iupac_name'].isna().sum()
number_of_iupac = df_old_less_iupac['iupac_name'].notna().sum()

num_no_iupac_but_has_cid = (df_old_less_iupac['iupac_name'].isna() & df_old_less_iupac['cid'].notna()).sum()

df_new_iupacs = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_now_with_iupac_from_pubchem.csv')

new_iupacs = df_new_iupacs['iupac_name'].notna().sum()



out = f'Total coconut compounds: {total_compounds} \nNumber of iupac missinng from origional coconut download: {number_of_iupac_missing}\n Number of iupac in og: {number_of_iupac}\n Number of compounds in coconut og missing their iupac but had a cid: {num_no_iupac_but_has_cid}\n Number of new iupacs added from pubchem: {new_iupacs}'

print(out)

/tmp/ipykernel_12628/1765527528.py:3: DtypeWarning: Columns (0: np_classifier_is_glycoside) have mixed types. Specify dtype option on import or set low_memory=False.
  df_old_less_iupac = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv')


Total coconut compounds: 738827 
Number of iupac missinng from origional coconut download: 74593
 Number of iupac in og: 664234
 Number of compounds in coconut og missing their iupac but had a cid: 54988
 Number of new iupacs added from pubchem: 52469


## now adding the new iupacs to a new file getting ready for DB insertion (still need to try opsin)

In [2]:
og  = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv')
new = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_now_with_iupac_from_pubchem.csv')


iupac_map = new.set_index('cid')['iupac_name']

og['iupac_name'] = og['iupac_name'].fillna(og['cid'].map(iupac_map))

og.to_csv('/home/school/masters/Scripts/coconut_full/coconut_with_pubchem_iupacs.csv', index=False)



/tmp/ipykernel_12628/2230518809.py:1: DtypeWarning: Columns (0: np_classifier_is_glycoside) have mixed types. Specify dtype option on import or set low_memory=False.
  og  = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv')


InvalidIndexError: Reindexing only valid with uniquely valued Index objects

# the above code snippet cant work becauase there are cid duplicates
- trying to figure this out now

In [7]:
import pandas as pd
og  = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', low_memory=False)
new = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_now_with_iupac_from_pubchem.csv')

og = og.dropna(subset='cid')
og[og.duplicated(subset='cid')]
num = og.duplicated(subset='cid').sum()

print('total dups:' + str(num) + '\n')
print(og['cid'])

total dups:14100

0          16400397.0
1            177785.0
2          10817089.0
3          39377583.0
4         145705321.0
             ...     
738821      6217354.0
738822      1778775.0
738823     16397128.0
738824      1590746.0
738826     16408156.0
Name: cid, Length: 636970, dtype: float64


## the plan now is to deduplicate the new iupacs file so i can add it to the others (I am assuming things with the same cid by definition have the same iupc name)

In [8]:
og  = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv')
new = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_now_with_iupac_from_pubchem.csv')


new = new.dropna(subset='cid')
new = new.drop_duplicates(subset='cid', keep='first')
iupac_map = new.set_index('cid')['iupac_name']

og['iupac_name'] = og['iupac_name'].fillna(og['cid'].map(iupac_map))

og.to_csv('/home/school/masters/Scripts/coconut_full/coconut_with_pubchem_iupacs.csv', index=False)

/tmp/ipykernel_15073/901367274.py:1: DtypeWarning: Columns (0: np_classifier_is_glycoside) have mixed types. Specify dtype option on import or set low_memory=False.
  og  = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv')


## Sorting out the deduplication issue 

In [ ]:
import pandas as pd

df = pd.read_csv('coconut_full/coconut_with_pubchem_iupacs.csv')

df = df[['identifier','standard_inchi','standard_inchi_key','name','iupac_name','annotation_level','np_likeness','collections','dois','synonyms','cas','coconut_id','cid']]

inchi_set = set()
new_df = []

counter = 0
for row in df.itertuples():
    inchi = None
    inchi_key = None
    name = None
    iupac = None
    annotation_level = None
    np_likeness = None
    collections = None
    dois = None
    synonyms = None
    cas = None
    coconut_id = None
    cid = None

    if row.standard_inchi not in inchi_set:
        inchi = row.standard_inchi
        inchi_key = row.standard_inchi_key
        inchi_set.add(row.standard_inchi)
        dupes = df.loc(df['standard_inchi']==inchi)

        for line in dupes.itertuples():
            if line.name not in name:
                name.append(line.name)
            if line.iupac_name not in iupac:
                iupac.append(line.iupac_name)
            if line.annotation_level not in annotation_level:
                annotation_level.append(line.annotation_level)
            if line.np_likeness not in np_likeness:
                np_likeness.append(line.np_likeness)
            if line.collections not in collections:
                collections.append(line.collections)
            if line.dois not in dois:
                dois.append(line.dois)
            if line.synonyms not in synonyms:
                synonyms.append(line.synonyms)
            if line.cas not in cas:
                cas.append(line.cas)
            if line.coconut_id not in coconut_id:
                coconut_id.append(line.coconut_id)
            if line.cid not in cid:
                cid.append(line.cid)
    name = name.to_list('|')
    iupac = iupac.to_list('|')
    annotation_level = annotation_level.to_list('|')
    np_likeness = np_likeness.to_list('|')
    collections = collections.to_list('|')
    dois = dois.to_list('|')
    synonyms = synonyms.to_list('|')
    cas = cas.to_list('|')
    coconut_id = coconut_id.to_list('|')
    cid = cid.to_list('|')

    new_df.append([row.identifier, inchi, inchi_key, name, iupac, annotation_level, np_likeness, collections, dois, synonyms, cas, coconut_id, cid])

df = pd.DataFrame(new_df, columns=['identifier','standard_inchi','standard_inchi_key','name','iupac_name','annotation_level','np_likeness','collections','dois','synonyms','cas','coconut_id','cid'])
df.to_csv('coconut_full/coconut_with_pubchem_iupacs_no_dupes.csv', index=False)



## I am keeping the above but It was never run. It is apparently very slow and gropby is supposed to be better so I am going to try that instead on my own and then let AI help me fix if it is dodgy

In [1]:
import pandas as pd

def neat_list(strrr):
    out = []
    for strr in strrr:
        if pd.notna(strr):
            out.append(str(strr).strip())
    return '|'.join(out)    

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_pubchem_iupacs.csv',low_memory=False)

df = df[['identifier','standard_inchi','standard_inchi_key','name','iupac_name','annotation_level','np_likeness','collections','dois','synonyms','cas','coconut_id','cid']]

df_new = df.groupby('standard_inchi',as_index=False,sort=False).agg({'name' : neat_list,
                                                                    'iupac_name': neat_list,
                                                                    'annotation_level': neat_list,
                                                                    'np_likeness': neat_list,
                                                                    'collections': neat_list,
                                                                    'dois': neat_list,
                                                                    'synonyms': neat_list,
                                                                    'cas': neat_list,
                                                                    'coconut_id': neat_list,
                                                                    'cid': neat_list})


df_new.to_csv('/home/school/masters/Scripts/coconut_full/coconut_iupacs_inchi_deduplicated.csv', index=False)

KeyboardInterrupt: 

# 9 Sep


## sorting out instances where the | combination added duplicates within columns

In [ ]:
#####


# The below was never run see labbook there was a lot of drama


####
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_iupacs'_inchi_deduplicated.csv')

def duplicate_checker(input):
    arr = input.split('|')
    arr = [ar.strip() for ar in arr]
    arr_set = set(arr)

    if len(arr_set) == 1:
        return arr_set[0], None
    else:
        sortin = sorted(arr_set,key=len)
        best = sortin[0]
        synonyms = sortin[1:]
        return best, '|'.join(synonyms)

def synonym_adder(new,old):
    new = new + '|' + old
    new_list = []
    new_list = new.split('|')

    new_list = [ne.strip() for ne in new]
    new_set = set(new)
    return '|'.join(new_set)
    
for idx, row in df.iterrows():
    name = row.name
    if '|' in name:
        name, new_synonyms = duplicate_checker(name)
        df.at[idx,'name'] = name
        old_synonyms = row.synonyms
        df.at[idx,'synonyms'] = synonym_adder(old_synonyms,new_synonyms)
    if '|'
        

# 14 Sep


## Sorting out the ~{} in the coconut iupacs

In [2]:
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_iupacs_inchi_deduplicated.csv')

def clean_iupac(iupac):
    if isinstance(iupac, str) and '~{' in iupac:
        out = iupac.replace('~{', '')
        out = out.replace('}', '')
        return out
    return iupac

df['name'] = df['name'].apply(clean_iupac)
df['iupac_name'] = df['iupac_name'].apply(clean_iupac)
df['synonyms'] = df['synonyms'].apply(clean_iupac)

df.to_csv('/home/school/masters/Scripts/coconut_full/coconut_compounds_cleaned_iupac.csv',index=False)

/tmp/ipykernel_18701/1342524253.py:3: DtypeWarning: Columns (0: annotation_level, 1: np_likeness, 2: cid) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_iupacs_inchi_deduplicated.csv')


## Sorting out the | for the things

In [2]:
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_compounds_cleaned_iupac.csv')

def name_cleaner(names):
    if pd.isna(names):
        return '', ''
    naame = str(names).split('|')
    naame = [naam.strip() for naam in naame]
    naame = list(dict.fromkeys(naame))
    naam = naame[0]
    naame.remove(naam)
    synonyms = '|'.join(naame)

    return naam, synonyms

def deduplicate(names):
    if pd.isna(names):
        return ''
    
    naame = str(names).split('|')
    naame = [naam.strip() for naam in naame]
    naame = list(dict.fromkeys(naame))
    out = '|'.join(naame)
    return out


def row_cleaner(row):
    inchi = row.standard_inchi
    name, N_synonyms = name_cleaner(row.name)
    iupac, I_synonyms = name_cleaner(row.iupac_name)
    annotation_level = row.annotation_level
    np_likeness = row.np_likeness
    collections = deduplicate(row.collections)
    dois = deduplicate(row.dois)
    synonyms = deduplicate(row.synonyms)
    cas = deduplicate(row.cas)
    coconut_id = deduplicate(row.coconut_id)
    cid = deduplicate(row.cid)
    syns = I_synonyms + '|' + N_synonyms + '|' + synonyms
    synonyms = deduplicate(syns)

    return [inchi, name, iupac, annotation_level, np_likeness, collections, dois, synonyms, cas, coconut_id, cid]

entry = []
for row in df.itertuples():
    entry.append(row_cleaner(row))

columns = ['inchi','name','iupac', 'annotation_level', 'np_likeness', 'collections', 'dois', 'synonyms', 'cas', 'coconut_id','cid']

df_out = pd.DataFrame(entry, columns=columns)
df_out.to_csv('/home/school/masters/Scripts/coconut_full/compounds_deduplicated_infield.csv')


print(df_out[['name', 'iupac', 'synonyms']].head(10))

print('Blank names:', (df_out['name'] == '').sum())
print('Blank IUPAC names:', (df_out['iupac'] == '').sum())

print(
    'Synonym fields starting or ending with "|":',
    df_out['synonyms'].str.startswith('|').sum()
    + df_out['synonyms'].str.endswith('|').sum()
)

print(
    'Synonym fields containing "||":',
    df_out['synonyms'].str.contains(r'\|\|', regex=True).sum()
)


/tmp/ipykernel_34521/1092763442.py:3: DtypeWarning: Columns (0: annotation_level, 1: np_likeness, 2: cid) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_compounds_cleaned_iupac.csv')


                                                name  \
0                                          STL537548   
1                                         16791-35-8   
2                                                      
3  2-(7,8-dimethoxy-1-oxophthalazin-2(1H)-yl)-N-(...   
4                                          STL561968   
5  10-Benzyl-2,3-dihydroimidazo[2,1-b]quinazolin-...   
6                                          STL540362   
7                                          STL516784   
8                                          STL578747   
9                                          STL550334   

                                               iupac  \
0  (8S)-6-(4,4-diethoxybutyl)-2-(2-ethoxyphenyl)-...   
1                        undec-10-enehydroxamic acid   
2  [(1R,2R,3S,4S,4aR,5R,8R,8aR)-4-[(3aR,5S,6aS)-2...   
3  2-(7,8-dimethoxy-1-oxo-phthalazin-2-yl)-N-(3-m...   
4  2-[12-(carboxymethyl)-16,17-dihydroxy-5,13-bis...   
5  10-benzyl-2,3-dihydroimidazo[2,1-b]quinazoli

## removing the || and the synonyms starting with |

In [3]:
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/compounds_deduplicated_infield.csv')

def remove_pipes(value):
    if pd.isna(value):
        return ''

    while '||' in value:
        value = value.replace('||','|')

    value = value.strip('|')
    return value

df['synonyms'] = df['synonyms'].apply(remove_pipes)

df.to_csv('/home/school/masters/Scripts/coconut_full/compounds_deduplicated_infield.csv', index=False)


print('Blank names:', (df['name'] == '').sum())
print('Blank IUPAC names:', (df['iupac'] == '').sum())

print(
    'Synonym fields starting or ending with "|":',
    df['synonyms'].str.startswith('|').sum()
    + df['synonyms'].str.endswith('|').sum()
)

print(
    'Synonym fields containing "||":',
    df['synonyms'].str.contains(r'\|\|', regex=True).sum()
)

/tmp/ipykernel_34521/1211826756.py:3: DtypeWarning: Columns (0: annotation_level, 1: np_likeness, 2: cid) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/school/masters/Scripts/coconut_full/compounds_deduplicated_infield.csv')


Blank names: 0
Blank IUPAC names: 0
Synonym fields starting or ending with "|": 0
Synonym fields containing "||": 0


## Adding SMILES

In [5]:
from rdkit import Chem
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/compounds_deduplicated_infield.csv')
df['smiles'] = None
df['isometric_smiles'] = None

for idx, row in df.iterrows():
    inchi = row.inchi
    mol = Chem.MolFromInchi(inchi)
    
    smiles = Chem.MolToSmiles(mol, isomericSmiles=False,canonical=True)    
    iso_smiles = Chem.MolToSmiles(mol, isomericSmiles=True,canonical=True)

    df.at[idx,'smiles'] = smiles
    df.at[idx, 'isometric_smiles'] = iso_smiles


/tmp/ipykernel_34521/2864064430.py:4: DtypeWarning: Columns (0: annotation_level, 1: np_likeness, 2: cid) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/school/masters/Scripts/coconut_full/compounds_deduplicated_infield.csv')
[14:28:24] WARNING: not removing hydrogen atom without neighbors
[14:28:26] WARNING:  Problems/mismatches: Mobile-H( Hydrogens: Number; Mobile-H groups: Number; Charge(s): Do not match)

[14:28:30] Cannot assign bond directions!
[14:28:30] WARNING:  Problems/mismatches: Mobile-H( Hydrogens: Locations or number; Mobile-H groups: Falsely present, Attachment points)

[14:28:31] WARNING:  Problems/mismatches: Mobile-H( Hydrogens: Locations or number; Mobile-H groups: Falsely present, Attachment points)

[14:28:31] Cannot assign bond directions!
[14:28:33] WARNING:  Problems/mismatches: Mobile-H( Hydrogens: Locations or number; Mobile-H groups: Attachment points, Number)

[14:28:36] Cannot assign bond directions!
[14

ArgumentError: Python argument types in
    rdkit.Chem.rdmolfiles.MolToSmiles(NoneType)
did not match C++ signature:
    MolToSmiles(RDKit::ROMol mol, bool isomericSmiles=True, bool kekuleSmiles=False, int rootedAtAtom=-1, bool canonical=True, bool allBondsExplicit=False, bool allHsExplicit=False, bool doRandom=False, bool ignoreAtomMapNumbers=False)
    MolToSmiles(RDKit::ROMol mol, RDKit::SmilesWriteParams params)

# 15 Sep

## excluding coconut compounds

- as seen yesterday some of the compounds are weird so see labbook I am excluding some of the compounds RD kit cant make nice with

### Making list of compounds that only belong to a non plant


In [ ]:
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_plants/checklist_matched_organisms.tsv', sep = '\t',usecols= ['original_coconut_id','kingdom'], low_memory=False)
num = f'{len(df):,}'.replace(',',' ')
print(f'Number of coconut organism relationships: {num}')

df = df.loc[df['kingdom'].eq('Plantae')].copy()
num = f'{len(df):,}'.replace(',',' ')
print(f'Number of cocont plant relationships {num}')

df.drop_duplicates(subset='original_coconut_id', inplace=True)
num = f'{len(df):,}'.replace(',',' ')
print(f'Number of coconut compounds made by a plant {num}')

df.drop(columns='kingdom', inplace = True)

df.to_csv('/home/school/masters/Scripts/coconut_full/coconut_plants/coconut_compounds_made_by_plants_only.csv',index=False)

Number of coconut organism relationships: 1 267 088
Number of cocont plant relationships 937 333
Number of coconut compounds made by a plant 175 364


Writing the script for wonko noe that will remove anything that:
- does not have a coconutID in coconut_compounds_made_by_plants_only.csv 
- has a nplikeness score below -1 
- then uses RD kit to try convert the inchi to smiles and anytime it has a warning it sends that compound to error file

>>>>>>>>>>>>>>>>>>>>>>>< file name on wonko: cleaning_coco_adding_smiles.py >

In [ ]:
import pandas as pd
from rdkit import Chem, rdBase
import nispo
import logging
from io import StringIO

plant_compounds_file = '/nlustre/users/nathanc/scratch/coconut_compounds_made_by_plants_only.csv'
coconut_compounds_file = '/nlustre/users/nathanc/scratch/compounds_deduplicated_infield.csv'
output_file = '/nlustre/users/nathanc/scratch/cleaned_coconut.csv'
error_file = '/nlustre/users/nathanc/scratch/errors.csv'

rdBase.LogToPythonLogger()

rdkit_logger = logging.getLogger('rdkit')
rdkit_logger.setLevel(logging.DEBUG)

log_stream = StringIO()
log_handler = logging.StreamHandler(log_stream)
log_handler.setFormatter(
    logging.Formatter('%(levelname)s: %(message)s')
)

rdkit_logger.handlers.clear()
rdkit_logger.addHandler(log_handler)
df_plant_compounds = pd.read_csv(plant_compounds_file, low_memory=False)
df_coco = pd.read_csv(coconut_compounds_file, low_memory=False)

plant_set = set(df_plant_compounds['original_coconut_id'].astype(str))

def check_plant(ids):
    ids = str(ids).split('|')
    for id in ids:
        if id.strip() in plant_set:
            return True
    return False

df_coco['from_plant'] = df_coco['coconut_id'].apply(check_plant)
df_coco_plants = df_coco.loc[df_coco['from_plant']].copy()

num_plant_compounds = f'{len(df_coco_plants):,}'.replace(',', ' ')

def is_likely_np(likeness):
    likenesses = likeness.split('|')

    likenesses = [float(like) for like in likenesses]

    best_case = max(likenesses)
    if best_case < -1.0:
        return False
    else:
        return True

df_coco_plants['likely_np'] = df_coco_plants['np_likeness'].apply(is_likely_np)
df_coco_plants = df_coco_plants.loc[df_coco_plants['likely_np']].copy()

num_likely_np = f'{len(df_coco_plants):,}'.replace(',', ' ')

errors = []

def get_smiles(inchi):
    log_stream.seek(0)
    log_stream.truncate(0)
    try:
        mol = Chem.MolFromInchi(inchi, treatWarningAsError=True)
        if mol is None:
            raise ValueError('RDKit could not create a molecule from this InChI.')
        c_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)
        i_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
        rdkit_messages = log_stream.getvalue().strip()
        if rdkit_messages:
            errors.append([
                inchi,
                rdkit_messages
            ])
            return None, None
        return c_smiles, i_smiles
    except Exception as e:
        e = str(e)
        error = f'{inchi} has an error: \n{e}'
        errors.append([inchi,error])
        return None, None

count = 0
df_len = len(df_coco_plants)
for idx, row in df_coco_plants.iterrows():
    count += 1
    inchi = row.inchi
    c_smiles,i_smiles = get_smiles(inchi)
    df_coco_plants.at[idx,'canonical_smiles'] = c_smiles
    df_coco_plants.at[idx,'isomeric_smiles'] = i_smiles
    if count % 10000 == 0 or count == df_len or count == 1:
        prog = int(count/(df_len)*100)
        print(f'Progress: {prog}%', flush=True)
    
df_errors = pd.DataFrame(errors, columns = ['inchi','error'])
df_errors.to_csv(error_file, index=False)

df_coco_plants.to_csv(output_file, index=False)

num_at_end = f'{len(df_coco_plants):,}'.replace(',', ' ')
print(f'There are {num_plant_compounds} plant compounds \nThere are {num_likely_np} likely np compounds (threshold above -1) \n There are at the end:{num_at_end}')
num_smiles_missing = f'{(df_coco_plants["canonical_smiles"].isna().sum()):,}'.replace(',', ' ')
print(f'There are {num_smiles_missing} missing smiles')

# 16 Sep

# (Lentedag!!!!!)

## cheking out the errors from the cleaning should mainly be RDkit parsing smiles drawing things

In [5]:
import pandas as pd
#removing the enchi to see unique error messages
df = pd.read_csv('/home/school/masters/Scripts/coconut_full/errors_15_sep_cleaning_coconut.csv')
errs = []
for idx, row in df.iterrows():
    inchi = row.inchi
    error = row.error

    errs.append(error)

for er in errs:
    print(er+ '\n')



InChI=1S/C25H20O17/c26-9-4-12(28)10-6-15(20(39-14(10)5-9)8-1-2-11(27)13(29)3-8)40-25-19(32)18(31)17(30)16(41-25)7-38-23(36)24(37)42-22(35)21(33)34/h1-6,16-19,25,30-32H,7H2,(H4-,26,27,28,29,33,34,36,37)/p+1/t16-,17-,18+,19+,25-/m1/s1 has an error: 
(<rdkit.Chem.rdchem.Mol object at 0x7f133d8cbf40>, ' Problems/mismatches: Mobile-H( Mobile-H groups: Attachment points, Number)')

InChI=1S/C18H20O5/c1-6-10(2)17(21)22-15(16-18(4,5)23-16)13-9-12(11(3)19)7-8-14(13)20/h6-9,20H,1-5H3/b10-6- has an error: 
(<rdkit.Chem.rdchem.Mol object at 0x7f133d72bf40>, ' Problems/mismatches: Mobile-H( Stereobonds/cumulenes: Extra undefined)')

InChI=1S/C18H27O5/c1-17(2)7-4-8-18(3)11(12(20)5-6-15(17)18)9-13(21)16(23)14(22)10-19/h14,19,21-22H,4-10H2,1-3H3/t14?,18-/m1/s1 has an error: 
(<rdkit.Chem.rdchem.Mol object at 0x7f133d663f40>, ' Problems/mismatches: Mobile-H( Hydrogens: Locations or number; Mobile-H groups: Falsely present, Attachment points)')

InChI=1S/C5H13NO4S/c1-6(2,3)4-5-10-11(7,8)9/h4-5H2,1-3H3 

## Adding missing smiles and removing H2

In [8]:
import pandas as pd
from rdkit import Chem

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut.csv')

df = df[df["inchi"] != "InChI=1S/H2/h1H"].copy()

for idx, row in df.iterrows():
    c_smiles = row.canonical_smiles
    inchi = row.inchi
    if pd.isna(c_smiles):
        mol = Chem.MolFromInchi(inchi)
        if mol:
            c_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)
            i_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
            df.at[idx,'canonical_smiles'] = c_smiles
            df.at[idx,'isomeric_smiles'] = i_smiles

df.to_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut_all_smiles.csv',index=False)

/tmp/ipykernel_8264/3114018064.py:4: DtypeWarning: Columns (0: cid) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut.csv')
[11:39:57] WARNING:  Problems/mismatches: Mobile-H( Mobile-H groups: Attachment points, Number)

[11:39:58] WARNING:  Problems/mismatches: Mobile-H( Stereobonds/cumulenes: Extra undefined)

[11:39:58] WARNING:  Problems/mismatches: Mobile-H( Hydrogens: Locations or number; Mobile-H groups: Falsely present, Attachment points)

[11:39:58] Explicit valence for atom # 10 S, 7, is greater than permitted
[11:39:58] ERROR: Explicit valence for atom # 10 S, 7, is greater than permitted

[11:39:58] WARNING:  Problems/mismatches: Mobile-H( Hydrogens: Number; Mobile-H groups: Number; Charge(s): Do not match)

[11:39:59] WARNING:  Problems/mismatches: Mobile-H( Hydrogens: Number; Mobile-H groups: Number; Charge(s): Do not match)

[11:39:59] Explicit valence for atom # 19 S, 7

adding the 3 weird compounds the rdkit could not parse

In [10]:
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut_all_smiles.csv') 

idx = df.index[df["inchi"].eq("InChI=1S/C5H13NO4S/c1-6(2,3)4-5-10-11(7,8)9/h4-5H2,1-3H3")]
df.loc[idx,'canonical_smiles'] = 'C[N+](C)(C)CCOS(=O)(=O)[O-]'
df.loc[idx,'isomeric_smiles'] = 'C[N+](C)(C)CCOS(=O)(=O)[O-]'

idx = df.index[df["inchi"].eq("InChI=1S/C9H18O9S2/c10-1-7(18-20(15,16)17)5(12)3-19-4-6(13)9(14)8(19)2-11/h5-14H,1-4H2/t5-,6-,7+,8-,9+,19?/m1/s1")]
df.loc[idx,'canonical_smiles'] = 'O=S(=O)([O-])OC(CO)C(O)C[S+]1CC(O)C(O)C1CO'
df.loc[idx,'isomeric_smiles'] = 'O=S(=O)([O-])O[C@@H](CO)[C@H](O)C[S+]1C[C@@H](O)[C@H](O)[C@H]1CO'

idx = df.index[df["inchi"].eq("InChI=1S/C12H24O12S2/c13-1-5(15)10(19)11(20)12(24-26(21,22)23)7(17)4-25-3-6(16)9(18)8(25)2-14/h5-20H,1-4H2/t5?,6-,7?,8-,9+,10?,11?,12?,25?/m1/s1")]
df.loc[idx,'canonical_smiles'] = 'O=S(=O)([O-])OC(C(O)C[S+]1CC(O)C(O)C1CO)C(O)C(O)C(O)CO'
df.loc[idx,'isomeric_smiles'] = 'O=S(=O)([O-])OC(C(O)C[S+]1C[C@@H](O)[C@H](O)[C@H]1CO)C(O)C(O)C(O)CO'



df.to_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut_all_smiles.csv',index=False)

/tmp/ipykernel_8264/3330853632.py:3: DtypeWarning: Columns (0: cid) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut_all_smiles.csv')


## removing smiles redundancy for 2d compounds

In [13]:
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut_all_smiles.csv') 

df.drop(columns='isometric_smiles',inplace=True)

for idx, row in df.iterrows():
    c_smiles = row.isomeric_smiles
    i_smiles = row.canonical_smiles

    if c_smiles.strip() == i_smiles.strip():
        df.at[idx,'isomeric_smiles'] = None



df.to_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut_all_smiles.csv',index=False)

/tmp/ipykernel_8264/3661661962.py:3: DtypeWarning: Columns (0: cid) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut_all_smiles.csv')


## Code for wonko to assign nispo iupac to empty iupac

In [ ]:
import nispo
import pandas as pd
from rdkit import Chem

input_file = '/nlustre/users/nathanc/scratch/cleaned_coconut_all_smiles.csv'
output_file = '/nlustre/users/nathanc/scratch/cleaned_coconut_all_iupac.csv'
error_file = '/nlustre/users/nathanc/scratch/errors_for_cleaned_coconut_iupac.csv'

df = pd.read_csv(input_file)
errors = []
for idx, row in df.iterrows():
    iupac = row.iupac
    if pd.isna(iupac):
        inchi = row.inchi
        iso_smiles = row.isomeric_smiles
        can_smiles = row.canonical_smiles

        if pd.isna(iso_smiles):
            smiles = can_smiles
        else:
            smiles = iso_smiles

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            mol = Chem.MolFromInchi(inchi)
            if mol is None:
                e = 'unable to make mol'
                errors.append([inchi,e])
                continue
        try:
            iupac = nispo.mol_to_iupac(mol)
            df.at[idx,'iupac'] = iupac
        except Exception as e:
            e = str(e)
            errors.append([inchi,e])

df_err = pd.DataFrame(errors, columns = ['inchi', 'error'])
df_err.to_csv(error_file, index=False)
df.to_csv(output_file,index= False)    

# 17 Sep

## Sorting out the names by adding the synonym to name everytime there is only one synonym

In [2]:
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut_all_iupac.csv') 

for idx, row in df.iterrows():
    name = row['name']
    synonyms = row['synonyms']

    if pd.isna(name) and pd.notna(synonyms):
        syns = synonyms.split('|')
        if len(syns) < 2:
            df.at[idx,'name'] = synonyms
            df.at[idx,'synonyms'] = None

df.to_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut_all_iupac.csv',index=False)

/tmp/ipykernel_7679/17325252.py:3: DtypeWarning: Columns (0: cid) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut_all_iupac.csv')


## Adding inchi_keys

In [5]:
import pandas as pd
from rdkit import Chem

def add_key(inchi):
    inchi_key = Chem.inchi.InchiToInchiKey(inchi)
    return inchi_key


df = pd.read_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut_all_iupac.csv') 

df['inchi_key'] = [add_key(inchi) for inchi in df['inchi']]

df.to_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut_all_iupac.csv',index=False)

/tmp/ipykernel_7679/2871837518.py:9: DtypeWarning: Columns (0: cid) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut_all_iupac.csv')


## Making nice for DB

In [8]:
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut_all_iupac.csv') 

df.drop(columns=['Unnamed: 0'], inplace=True)
df.drop(columns=['from_plant','likely_np','annotation_level','np_likeness'], inplace=True)


df['compound_id'] = [count+1 for count in range(len(df))]

df = df[['compound_id','name','inchi','inchi_key','canonical_smiles','isomeric_smiles','iupac','synonyms','collections','dois','coconut_id','cid','cas']]

df.to_csv('/home/school/masters/Scripts/coconut_full/full_coconut_for_DB.csv',index=False)

/tmp/ipykernel_7679/1061169332.py:3: DtypeWarning: Columns (0: cid) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/school/masters/Scripts/coconut_full/cleaned_coconut_all_iupac.csv')


## Looking good I just want to make sure there is no entry that has no name, synonym or iupac

In [12]:
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/full_coconut_for_DB.csv')
count = 0
for idx, row in df.iterrows():
    name = row['name']
    iupac = row['iupac']
    synonyms = row['synonyms']
    
    if pd.isna(name) and pd.isna(iupac) and pd.isna(synonyms):
        print(row.compound_id)
        count = count + 1

print(count)

148
194
308
577
595
597
601
685
818
1190
1269
1705
1965
1967
1975
1982
1983
2079
3122
3134
3389
3396
3397
3406
3486
3938
3972
4173
4330
4591
4848
4862
5414
5985
6231
6235
6313
6320
6322
6698
6822
7180
7335
7608
8209
8614
8737
8750
9087
9423
9624
9665
9902
10135
10283
10373
10389
10390
10391
10475
10596
10636
11413
11567
11639
11783
11807
11818
11826
11828
11831
11835
12364
12412
13292
14156
14319
14565
14696
14800
15065
15141
15521
15665
15909
16029
16055
17000
17036
17372
17373
17384
17464
18222
18230
18345
18386
18463
18658
18734
19079
19212
19755
20033
20099
20100
20517
21009
21391
21497
21565
22409
22485
22572
22681
22854
22859
22860
22876
23504
23732
23841
23845
24207
24314
25319
25327
25590
26069
26079
26490
26588
26605
26760
26914
27018
27021
27043
27101
28313
28401
29449
29685
29687
29753
29756
29761
30983
31027
31087
31222
31240
32103
32339
32349
32355
32359
32363
32420
32550
33102
33419
33631
33741
33742
34162
34296
35141
36150
36208
36222
36260
36308
36446
36539
36547
36616


## checking lengths of things for DB varchar

In [16]:
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/full_coconut_for_DB.csv')

for col in df.columns:
    print(col, df[col].astype('string').str.len().max())

compound_id 6
name 1770
inchi 2513
inchi_key 27
canonical_smiles 839
isomeric_smiles 1526
iupac 3055
synonyms 212821
collections 1111
dois 67951
coconut_id 77
cid 21
cas 263
